# BME688 Gasklassifikation -- Trainings-Notebook (Entwurf)

Einfache Baseline-Pipeline für die aus `bme688_to_ei.py` / der Streamlit-App
exportierten Feature-Vektoren. Ziel ist ein erster Klassifikator zur
Einschätzung, wie gut sich die vier Klassen anhand der Gaswiderstands-Features
trennen lassen -- bevor ihr das Ganze in Edge Impulse für die
Embedded-Deployment-Pipeline weiterverwendet.

**Wichtigstes Prinzip in diesem Notebook:** Der Train/Test-Split erfolgt
immer auf Ebene ganzer Mess-Sessions, nie auf Ebene einzelner Zeilen/Vektoren.
Warum das entscheidend ist, steht in Abschnitt 2.

## 1. Daten laden

Ladet hier eine mit `--combine label` oder `--combine all` (ohne
`--apply-split`, also `apply_split=False`) erzeugte CSV -- die enthält
mehrere Zeilen (= Vektoren) mit den Spalten `vector_id`, `label`, `session`,
`category` (hier noch `"unsplit"`), `n_imputed` und den Feature-Spalten.

Falls ihr mehrere Label-Dateien habt (eine je Klasse), einfach alle laden
und konkatenieren.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedGroupKFold

# core.py muss im selben Ordner liegen (oder in sys.path aufgenommen werden)
sys.path.insert(0, str(Path.cwd()))
from core import train_test_split_by_session

DATA_DIR = Path("/home/tun/Projects/tuc/ML/Forschungspraktikum/CSV/transformiert/Badset/level2_per_profile/")  # ggf. anpassen

# Alle CSVs in diesem Ordner einlesen und zusammenführen
csv_files = list(DATA_DIR.glob("*.csv"))
assert csv_files, f"Keine CSVs in {DATA_DIR} gefunden -- Pfad anpassen."
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

meta_cols = ["vector_id", "label", "session", "category", "n_imputed",
             "sensor_index", "heater_profile_id", "cycle_id"]
feature_cols = [c for c in df.columns if c not in meta_cols]

print(df.shape)
print("Klassen:", df["label"].unique())
print("Sessions je Klasse:\n", df.groupby("label")["session"].nunique())
df.head()

## 2. Warum session-basiert splitten?

Innerhalb einer Mess-Session sind aufeinanderfolgende Zyklen stark
korreliert (gleiche Umgebungsbedingungen, gleiche Sensordrift, gleiches
Rauschmuster). Ein zufälliger Split auf Zeilenebene würde Zeilen derselben
Session sowohl im Training als auch im Test landen lassen -- das Modell
"erkennt" dann teilweise nur die Session wieder, nicht das eigentliche
Gasereignis. Ergebnis: die Testgenauigkeit sieht deutlich besser aus, als
das Modell auf echten, neuen Messungen tatsächlich leisten würde
(Data Leakage).

**Regel:** Eine Session gehört immer komplett entweder zu Training ODER
zu Test, nie zu beiden. Das übernimmt `train_test_split_by_session` aus
`core.py` automatisch -- dieselbe Funktion, die auch in der Streamlit-App
und der CLI genutzt wird.

In [ ]:
df_split = train_test_split_by_session(
    df, test_ratio=0.25, seed=7,   # nach Bedarf anpassen und neu ausführen
)

# Sanity-Check: keine Session in beiden Kategorien, Verhältnis je Klasse plausibel
check = df_split.groupby(["label", "category"])["session"].nunique().unstack(fill_value=0)
print(check)

overlap = (set(df_split.loc[df_split.category == "training", "session"]) &
           set(df_split.loc[df_split.category == "testing", "session"]))
assert not overlap, f"Leakage! Sessions in beiden Kategorien: {overlap}"
print("OK, keine Session in Training UND Test.")

train_df = df_split[df_split.category == "training"]
test_df  = df_split[df_split.category == "testing"]
print(f"Training: {len(train_df)} Vektoren aus {train_df.session.nunique()} Sessions")
print(f"Test:     {len(test_df)} Vektoren aus {test_df.session.nunique()} Sessions")

## 3. Features skalieren

Der `StandardScaler` wird **nur auf den Trainingsdaten gefittet** und dann
auf Test angewendet -- sonst fließt Information aus dem Test in die
Skalierung ein (ebenfalls eine Form von Leakage, auch wenn subtiler als
der Session-Fehler oben).

In [ ]:
X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test  = test_df[feature_cols].values
y_test  = test_df["label"].values

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

## 4. Baseline-Modell trainieren

Ein Random Forest ist ein robuster, schnell trainierter Startpunkt für
tabellarische Gassensor-Features und braucht kein Feature-Scaling
(hier trotzdem angewendet, falls ihr später zu einem linearen Modell oder
einem MLP wechseln wollt).

In [ ]:
clf = RandomForestClassifier(
    n_estimators=300, max_depth=None, class_weight="balanced",
    random_state=42, n_jobs=-1,
)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
print(classification_report(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, xticks_rotation=45)
plt.tight_layout()
plt.show()

## 5. Robustere Schätzung: Cross-Validation über Sessions

Ein einzelner Split ist bei wenigen Sessions je Klasse ziemlich
"zufallsabhängig" -- je nachdem, welche Session zufällig im Test landet,
schwankt die Genauigkeit stark. `StratifiedGroupKFold` macht dasselbe wie
oben (Gruppierung nach Session, keine Session in mehreren Folds), aber über
mehrere Folds hinweg gemittelt -- das gibt ein deutlich stabileres Bild als
ein einzelner Split, gerade bei noch überschaubarer Session-Anzahl.

In [ ]:
X = df[feature_cols].values
y = df["label"].values
groups = df["session"].values

n_splits = min(5, df.groupby("label")["session"].nunique().min())
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

scores = []
for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
    scaler_cv = StandardScaler().fit(X[train_idx])
    clf_cv = RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    )
    clf_cv.fit(scaler_cv.transform(X[train_idx]), y[train_idx])
    acc = clf_cv.score(scaler_cv.transform(X[test_idx]), y[test_idx])
    scores.append(acc)
    print(f"Fold {fold}: Accuracy = {acc:.3f}  "
          f"(Test-Sessions: {sorted(set(groups[test_idx]))})")

print(f"\nMittlere Accuracy: {np.mean(scores):.3f} +/- {np.std(scores):.3f}")

## 6. Nächste Schritte

- **Feature Importance** (`clf.feature_importances_`) je Heizstufe
  anschauen -- zeigt, welche Stufen für die Klassentrennung am wichtigsten
  sind. Kann helfen, das Heizprofil selbst zu optimieren.
- Modelle für **Level 2/3** (Sensorpaar bzw. alle Sensoren kombiniert)
  genauso durchspielen und vergleichen, ob die zusätzliche Information die
  Genauigkeit rechtfertigt -- gegen mehr Rechenaufwand/Overfitting-Risiko
  bei kleinen Datensätzen abwägen.
- Für die finale Embedded-Deployment-Pipeline: Datensatz (mit demselben
  session-sauberen Split!) in Edge Impulse hochladen und dort einen
  Klassifikator im Impulse Designer trainieren/quantisieren -- dieses
  Notebook dient primär der schnellen Voreinschätzung und Feature-Analyse.